# Resume-to-Job-Description Matcher — Siamese GRU

Loads the `resume_jd_pairs.csv` built in `ResumeJD_Logistic_Regression.ipynb` and trains a
**Siamese GRU** network: both the resume and the job description are encoded through
the *same* shared embedding + GRU branch, then the two encodings are combined and
passed to a small classifier head that outputs match probability.

In [1]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GRU, Dense, Dropout, Lambda, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


## 1. Load the pairs dataset

In [2]:
pairs = pd.read_csv(r"C:\Users\HP\Downloads\resume_jd_pairs.csv")
print("Shape:", pairs.shape)
pairs.head(3)


Shape: (144, 3)


,resume,jd,label
0,experienced cloud engineer with years of exper...,looking for a talented hr business partner wit...,0
1,results driven data analyst skilled in pandas ...,join us as a quality assurance analyst must ha...,0
2,experienced senior java engineer with years of...,we are hiring a backend developer to join our ...,1


In [3]:
pairs.duplicated().sum()


np.int64(0)

In [4]:
pairs.drop_duplicates(inplace=True)
pairs.dropna(inplace=True)


## 2. Train/test split

In [5]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    pairs[["resume", "jd"]], pairs["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=pairs["label"]
)


## 3. Prepare Sequences for the Neural Network

### Tokenization
A single shared tokenizer is fit on **both** resumes and JDs from the training set, so the
resume branch and JD branch of the Siamese network draw from the same vocabulary.

### Padding
Resumes are longer documents than job descriptions, so we use separate max lengths:
`MAX_LEN_RESUME=300`, `MAX_LEN_JD=150`.

In [6]:
MAX_LEN_RESUME = 300
MAX_LEN_JD = 150
VOCAB_SIZE = 15000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(pd.concat([X_train_text["resume"], X_train_text["jd"]]))

def encode(df):
    resume_seq = tokenizer.texts_to_sequences(df["resume"])
    jd_seq = tokenizer.texts_to_sequences(df["jd"])
    resume_pad = pad_sequences(resume_seq, maxlen=MAX_LEN_RESUME, padding="post", truncating="post")
    jd_pad = pad_sequences(jd_seq, maxlen=MAX_LEN_JD, padding="post", truncating="post")
    return resume_pad, jd_pad

X_train_resume, X_train_jd = encode(X_train_text)
X_test_resume, X_test_jd = encode(X_test_text)

y_train_arr = y_train.values
y_test_arr = y_test.values


## 4. Build the Siamese GRU model

Both branches share the **same** `Embedding` + `GRU` layers (true Siamese weight
sharing), so the network learns one text encoder used for both resumes and job descriptions.
The two encoded vectors are combined via absolute difference *and* concatenation before the
classifier head — the same merge strategy used for the QQP duplicate-question Siamese
networks.

In [7]:
embedding_dim = 64
rnn_units = 64

# Shared layers (weight-tied across both branches)
shared_embedding = Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, mask_zero=True)
shared_rnn = GRU(rnn_units)

resume_input = Input(shape=(MAX_LEN_RESUME,), name="resume_input")
jd_input = Input(shape=(MAX_LEN_JD,), name="jd_input")

resume_encoded = shared_rnn(shared_embedding(resume_input))
jd_encoded = shared_rnn(shared_embedding(jd_input))

abs_diff = Lambda(lambda x: tf.abs(x[0] - x[1]))([resume_encoded, jd_encoded])
merged = Concatenate()([resume_encoded, jd_encoded, abs_diff])

x = Dense(64, activation="relu")(merged)
x = Dropout(0.3)(x)
x = Dense(32, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

gru_model = Model(inputs=[resume_input, jd_input], outputs=output)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ resume_input (InputLayer)     │ (None, 300)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ jd_input (InputLayer)         │ (None, 150)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 150, 64)           │         960,000 │ resume_input[0][0],        │
│                               │                           │                 │ jd_input[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 300)               │               0 │ resume_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal_1 (NotEqual)        │ (None, 150)               │               0 │ jd_input[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gru (GRU)                     │ (None, 64)                │          24,960 │ embedding[0][0],           │
│                               │                           │                 │ not_equal[0][0],           │
│                               │                           │                 │ embedding[1][0],           │
│                               │                           │                 │ not_equal_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lambda (Lambda)               │ (None, 64)                │               0 │ gru[0][0], gru[1][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 192)               │               0 │ gru[0][0], gru[1][0],      │
│                               │                           │                 │ lambda[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 64)                │          12,352 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 64)                │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 32)                │           2,080 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 1)                 │              33 │ dense_1[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,425 (3.81 MB)

 Trainable params: 999,425 (3.81 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_gru = gru_model.fit(
    [X_train_resume, X_train_jd],
    y_train_arr,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.4854 - loss: 0.6935 - val_accuracy: 0.4167 - val_loss: 0.6928
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 516ms/step - accuracy: 0.4951 - loss: 0.6926 - val_accuracy: 0.4167 - val_loss: 0.6927
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step - accuracy: 0.5825 - loss: 0.6919 - val_accuracy: 0.4167 - val_loss: 0.6926
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 707ms/step - accuracy: 0.5631 - loss: 0.6918 - val_accuracy: 0.4167 - val_loss: 0.6926
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 605ms/step - accuracy: 0.5631 - loss: 0.6918 - val_accuracy: 0.4167 - val_loss: 0.6926
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 585ms/step - accuracy: 0.6214 - loss: 0.6899 - val_accuracy: 0.5000 - val_loss: 0.6925
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 592ms/step - accuracy: 0.6311 - loss: 0.6898 - val_accuracy: 0.5000 - val_loss: 0.6925
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 582ms/step - accuracy: 0.5825 - loss: 0.6895 - val_accuracy: 0.5000 - val_loss: 0.

## 5. Evaluate on the test set

In [ ]:
y_pred_gru = (gru_model.predict([X_test_resume, X_test_jd]) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred_gru))


In [ ]:
accuracy_score(y_test_arr, y_pred_gru)


In [ ]:
cm = confusion_matrix(y_test_arr, y_pred_gru)
ConfusionMatrixDisplay(cm, display_labels=["No Match", "Match"]).plot()
plt.title("Siamese GRU — Confusion Matrix")
plt.show()


In [ ]:
plt.plot(history_gru.history["accuracy"], label="train acc")
plt.plot(history_gru.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Siamese GRU — Training curve")
plt.show()


## 6. Save the trained model + tokenizer (used by `app.py`)

This is the best-performing branch of the three RNN variants, so its weights and the shared
tokenizer are what the Streamlit app loads at inference time.

In [ ]:
gru_model.save("resume_jd_gru_model.keras")

with open("resume_jd_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
